# One-dimensional quantile assignment

This notebook generates `fig:matching-1d-quantile-assignment`.  For two equal-weight empirical measures on the line,
$$
\alpha_n=\frac1n\sum_{i=1}^n\delta_{x_i},\qquad
\beta_n=\frac1n\sum_{j=1}^n\delta_{y_j},
$$
the optimal assignment for any convex cost $h(x-y)$ is obtained by sorting.  The figure removes all axes and keeps only the geometric information: smooth source/target laws, quantile samples, and the monotone connections between equal ranks.

In [ ]:
from pathlib import Path
import sys

for candidate in [Path.cwd(), Path.cwd() / "notebooks-figures", Path.cwd().parent / "notebooks-figures"]:
    if (candidate / "figure_style.py").exists():
        sys.path.insert(0, str(candidate.resolve()))
        break
else:
    raise RuntimeError("Could not locate figure_style.py")

import matplotlib.pyplot as plt
import numpy as np

from figure_style import BLUE, RED, interp_color, figure_dir, remove_axes, save_pdf, setup_matplotlib

setup_matplotlib()


## Mixtures and inverse-CDF samples

We use deterministic quantile levels $u_k=(k+1/2)/n$.  If $Q_\alpha$ and $Q_\beta$ denote the quantile functions, the matched atoms are
$$
x_k=Q_\alpha(u_k),\qquad y_k=Q_\beta(u_k).
$$
The second panel stresses the monotone rearrangement from one central Gaussian to a target with three modes of different widths and amplitudes.

In [ ]:
def normal_pdf(x, mean, std):
    return np.exp(-0.5 * ((x - mean) / std) ** 2) / (std * np.sqrt(2 * np.pi))


def mixture_pdf(x, weights, means, stds):
    pdf = np.zeros_like(x, dtype=float)
    for w, m, s in zip(weights, means, stds):
        pdf += w * normal_pdf(x, m, s)
    return pdf


def inverse_cdf_samples(grid, pdf, n):
    cdf = np.cumsum(pdf)
    cdf = (cdf - cdf[0]) / (cdf[-1] - cdf[0])
    levels = (np.arange(n) + 0.5) / n
    return np.interp(levels, cdf, grid), levels


def quantile_panel(alpha_params, beta_params, filename, n=52):
    grid = np.linspace(-3.3, 3.3, 5000)
    alpha_pdf = mixture_pdf(grid, **alpha_params)
    beta_pdf = mixture_pdf(grid, **beta_params)
    x_samples, levels = inverse_cdf_samples(grid, alpha_pdf, n)
    y_samples, _ = inverse_cdf_samples(grid, beta_pdf, n)

    fig, ax = plt.subplots(figsize=(3.1, 1.58))
    base_alpha = 0.76
    base_beta = 0.24
    scale = 0.27 / max(alpha_pdf.max(), beta_pdf.max())

    ax.fill_between(grid, base_alpha, base_alpha + scale * alpha_pdf, color=RED, alpha=0.16, linewidth=0)
    ax.plot(grid, base_alpha + scale * alpha_pdf, color=RED, lw=1.35)
    ax.fill_between(grid, base_beta, base_beta - scale * beta_pdf, color=BLUE, alpha=0.16, linewidth=0)
    ax.plot(grid, base_beta - scale * beta_pdf, color=BLUE, lw=1.35)

    for k, (xk, yk) in enumerate(zip(x_samples, y_samples)):
        ax.plot([xk, yk], [base_alpha, base_beta], color=interp_color(levels[k]), lw=0.48, alpha=0.43, zorder=1)

    ax.scatter(x_samples, np.full(n, base_alpha), s=10.0, color=RED, edgecolor="none", linewidth=0, zorder=3)
    ax.scatter(y_samples, np.full(n, base_beta), s=10.0, color=BLUE, edgecolor="none", linewidth=0, zorder=3)
    ax.set_xlim(grid.min(), grid.max())
    ax.set_ylim(-0.05, 1.05)
    remove_axes(ax)
    save_pdf(fig, out / filename, pad_inches=0.035)
    plt.close(fig)


## Exported panels

The first panel keeps the original two-mixture example, but with more samples and no axes.  The second panel shows a single central Gaussian transported to a three-component target mixture.

In [ ]:
fig_name = "matching-1d-quantile-assignment"
out = figure_dir(fig_name)

quantile_panel(
    alpha_params=dict(weights=[0.58, 0.42], means=[-2.05, -0.15], stds=[0.32, 0.48]),
    beta_params=dict(weights=[0.40, 0.60], means=[0.25, 1.75], stds=[0.36, 0.42]),
    filename="quantile-assignment.pdf",
)

quantile_panel(
    alpha_params=dict(weights=[1.0], means=[0.0], stds=[0.56]),
    beta_params=dict(weights=[0.50, 0.31, 0.19], means=[-1.78, 0.10, 1.72], stds=[0.25, 0.56, 0.31]),
    filename="central-to-three-modes.pdf",
)
